# Historical experiment notebook
This notebook preserves its original protocol and embedded source. For offline result reproduction use `scripts/reproduce.py` in the repository. For a new measurement keep the original experiment identity and do not overwrite old results. Archive checks are intentionally strict.


# Research Step 1 — Build a latency cost model

**Continue the existing CPU–GPU GPT-2 project without modifying its engine.**

This increment audits the completed `main/` sweep and fits two interpretable
latency predictors. It runs on **CPU only**. No GPU, checkpoint download, or
repeat of the original 45-case experiment is needed.

Input: upload **`run_20260919T162736Z_export.zip`**, your existing full-sweep
export—not the original project source ZIP or the later analysis ZIP.

Run the numbered sections in order. Stop if any assertion/test fails. The last
section downloads a result ZIP to share at the next implementation checkpoint.

**Scope:** this is retrospective model development. We have already examined
these measurements; the output is not a new blind-test result or an inference
speedup. Memory prediction and automatic placement are the next increment,
not part of this notebook.

## 1. Install the additive extension

This cell writes a separate `offloading_research_step1_v010/` directory and
installs only missing dependencies. It does not install, replace, or import
PyTorch. Existing files with different contents are not overwritten.

In [ ]:
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

requirements = {
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<3",
    "scipy": "scipy>=1.11,<2",
    "pytest": "pytest>=8,<10",
}
missing = [requirement for name, requirement in requirements.items()
           if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

WORK = (Path.cwd() if Path.cwd().name == "offloading_research_step1_v010"
        else Path.cwd() / "offloading_research_step1_v010")
WORK = WORK.resolve()
WORK.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE_B64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE_B64))) as z:
    for member in z.infolist():
        target = (WORK / member.filename).resolve()
        if not target.is_relative_to(WORK):
            raise RuntimeError("Unsafe source archive member")
        content = z.read(member)
        if target.exists() and target.read_bytes() != content:
            raise RuntimeError(f"Local edits found: {target}. Use a new work folder; edits will not be overwritten.")
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(content)
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("Extension installed in:", WORK)
print("Original inference files are unchanged. No GPU is required.")


## 2. Upload your existing full-sweep results

Choose **`run_20260919T162736Z_export.zip`** in the upload dialog. Only the archive's
`main/summary.csv`, `main/trials.csv`, and `main/metadata.json` will be read.

For local Jupyter, set the environment variable `OFFLOAD_BASELINE_ZIP` to the
archive's absolute path before running this cell. The archive is not extracted
and its contents are never executed.

In [ ]:
if os.environ.get("OFFLOAD_BASELINE_ZIP"):
    ARCHIVE = Path(os.environ["OFFLOAD_BASELINE_ZIP"]).expanduser().resolve()
else:
    try:
        from google.colab import files as colab_files
    except ImportError as exc:
        raise RuntimeError("For local Jupyter, set os.environ['OFFLOAD_BASELINE_ZIP'] to your archive path.") from exc
    print("Upload run_20260919T162736Z_export.zip")
    uploaded = colab_files.upload()
    names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(names) != 1:
        raise RuntimeError("Upload exactly one full-sweep export ZIP.")
    inputs = WORK / "inputs"
    inputs.mkdir(exist_ok=True)
    ARCHIVE = inputs / Path(names[0]).name
    payload = uploaded[names[0]]
    if ARCHIVE.exists() and ARCHIVE.read_bytes() != payload:
        raise RuntimeError("A different archive has this name. Rename the uploaded file first.")
    ARCHIVE.write_bytes(payload)
    del uploaded, payload
if not ARCHIVE.is_file():
    raise FileNotFoundError(ARCHIVE)
print("Using:", ARCHIVE)


## 3. Run the new software tests

These tests cover feature accounting, input validation, archive auditing,
workload-group separation, saved predictions, and extrapolation guards.
They are not new tests of pretrained GPT-2 logits or GPU performance.

In [ ]:
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "research_tests"],
    cwd=WORK, capture_output=True, text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
test_run.check_returncode()
TEST_LOG = test_run.stdout + "\n" + test_run.stderr


## 4. Audit, fit and perform retrospective grouped validation

The two model variants are `compute` and `compute_transfer`.

Each of nine folds withholds one complete **(batch, prompt length, output length)**
workload. All five GPU placements for that workload stay together. Feature
scaling and model fitting use only the remaining 40 rows.

Final models are then refitted to all 45 rows for later use. Their training-fit
errors are stored separately from cross-validation errors. Each row's target is
a median recomputed from all its trial measurements; slow trials are not deleted.

In [ ]:
from datetime import datetime, timezone

OUT = WORK / "outputs" / ("step1_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
subprocess.run([
    sys.executable, "-m", "offload_research.fit",
    "--archive", str(ARCHIVE),
    "--out", str(OUT),
], cwd=WORK, check=True)
(OUT / "test_output.txt").write_text(TEST_LOG, encoding="utf-8")
print("Checkpoint output:", OUT)


## 5. Inspect model errors

Lower error is better. **Do not call `100 - error` prediction accuracy.**
The worst-case column prevents a small median from hiding bad cases. These
historical folds are development diagnostics, not prospective validation.
The complete result file also contains mean and weighted errors.

In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads((OUT / "summary.json").read_text())
rows = []
for name, result in report["models"].items():
    cv = result["retrospective_grouped_cv"]
    rows.append({
        "model": name,
        "prefill median error %": cv["prefill_ms"]["median_ape_pct"],
        "decode median error %": cv["decode_tpot_ms"]["median_ape_pct"],
        "generation median error %": cv["generation_ms"]["median_ape_pct"],
        "generation worst error %": cv["generation_ms"]["worst_ape_pct"],
    })
display(pd.DataFrame(rows).round(2))
print("Verified configurations:", report["configuration_rows"])
print("Verified raw trials:", report["raw_trials_checked"])
print("Grouped folds per model:", report["folds_per_model"])
print("New GPU benchmarks run:", report["new_gpu_benchmarks_run"])


## 6. Predict a new workload without executing it

The example uses batch 3, prompt length 64, and 32 output tokens. This workload
combination was not in the main sweep, but lies inside its batch/prompt ranges.
These numbers are **predictions only**. Do not benchmark this workload yet:
we will first add memory prediction and fix the placement-evaluation protocol.

Generation is composed as:
`predicted first-token time + 31 * predicted average decode-step time`.

The fitting records refer to the original T4/two-thread environment—not to the
machine currently doing this CPU-only regression.

In [ ]:
from offload_research.cost_model import CostModel

predictions = []
for variant in ("compute", "compute_transfer"):
    model = CostModel.load(OUT / "models" / f"{variant}.json")
    for gpu_blocks in (0, 3, 6, 9, 12):
        prediction = model.predict(
            batch_size=3, sequence_length=64, new_tokens=32,
            gpu_layers=gpu_blocks,
        )
        predictions.append({"variant": variant, **prediction})
demo = pd.DataFrame(predictions)
demo.to_csv(OUT / "demo_predictions_NOT_MEASUREMENTS.csv", index=False)
display(demo[["variant", "gpu_layers", "predicted_ttft_ms",
              "predicted_decode_tpot_ms", "predicted_generation_ms", "measured"]].round(2))


## 7. Save this checkpoint

The result ZIP contains the calibrated models, fold assignments, per-case
prediction errors, software-test output, original hardware metadata, archive
hashes, and an implementation snapshot. It does not contain new GPU measurements
or copies of the model weights.

**Stop after this section.** Share the generated `step1_..._export.zip` for the
next checkpoint: adding memory prediction and budget-aware placement.

Source and method details are in `README_STEP1.md` in the work directory.

In [ ]:
import shutil

export_path = Path(shutil.make_archive(str(OUT) + "_export", "zip", root_dir=OUT))
print("Saved:", export_path)
try:
    from google.colab import files as colab_files
    colab_files.download(str(export_path))
except ImportError:
    print("Open the ZIP from your local notebook file browser.")
